# Notebook 1.1  First contact with Arabic ASR

**Companion to Chapter 1, *Introduction to Arabic Speech Technologies*.**

**Goal.** Run a pre-trained multilingual recognizer on Arabic audio, measure its error, and
see how text normalization changes the result. We then compare Modern Standard Arabic (MSA)
against dialect performance.

This notebook runs end-to-end with **no downloads**: if a real ASR model is not available it
falls back to realistic pre-recorded hypotheses, so the Word Error Rate (WER) and normalization
logic always run. In Google Colab you can enable the real model in the marked cell.

*Data:* [Common Voice Arabic](https://commonvoice.mozilla.org/ar) (openly licensed; report the
release version and validated hours when you use it).

## 1. Setup

In [ ]:
# In Colab, uncomment to enable the real recognizer and metrics library:
# !pip -q install transformers torchaudio jiwer soundfile librosa
import numpy as np

## 2. A tiny evaluation set

We use one MSA sample and three dialect samples. References are the correct transcripts.
In Colab, replace `audio_paths` with real Common Voice clips.

In [ ]:
# (Arabic reference, dialect label, optional audio path)
samples = [
    {'id': 'msa1',   'dialect': 'MSA',      'ref': 'ذَهَبَ الطَّالِبُ إِلَى المَدْرَسَةِ',     'audio': None},
    {'id': 'glf1',   'dialect': 'Gulf',     'ref': 'وش رايك نروح المكتبة بكرة',          'audio': None},
    {'id': 'egy1',   'dialect': 'Egyptian', 'ref': 'احنا رايحين الجامعة دلوقتي',          'audio': None},
    {'id': 'lev1',   'dialect': 'Levantine','ref': 'بدي روح ع السوق هلق',                'audio': None},
]
for s in samples:
    print(s['id'], '|', s['dialect'], '|', s['ref'])

## 3. Transcribe (real model if available, otherwise a realistic fallback)

The fallback hypotheses imitate the *kinds* of errors a strong MSA-trained model makes on
Arabic: it does well on MSA and degrades on dialect, often substituting the nearest MSA word
or dropping dialectal particles.

In [ ]:
def transcribe(samples):
    """Try a real Hugging Face pipeline; fall back to canned hypotheses."""
    try:
        from transformers import pipeline
        asr = pipeline('automatic-speech-recognition',
                       model='openai/whisper-small', generate_kwargs={'language': 'arabic'})
        hyps = {}
        for s in samples:
            if s['audio'] is None:
                raise RuntimeError('no audio paths set; using fallback')
            hyps[s['id']] = asr(s['audio'])['text']
        return hyps, 'whisper-small'
    except Exception as e:
        print('Using offline fallback hypotheses (', type(e).__name__, ')')
        return {
            'msa1': 'ذهب الطالب الى المدرسه',          # MSA: only spelling/diacritic differences
            'glf1': 'وش رأيك نروح المكتبة بكرة',         # Gulf: near-correct
            'egy1': 'نحن ذاهبون الجامعة الان',          # Egyptian -> pushed toward MSA
            'lev1': 'اريد ان اذهب الى السوق الان',       # Levantine -> rewritten as MSA
        }, 'offline-fallback'

hyps, model_name = transcribe(samples)
print('model:', model_name)
for s in samples:
    print(s['id'], '->', hyps[s['id']])

## 4. Metrics and an editable normalization step

In [ ]:
import re
# Arabic text normalization for scoring. Each rule is OPTIONAL and is a SCORING
# convention, not a claim of linguistic equivalence. Toggle the flags to see the
# effect on Word/Character Error Rate.
TASHKEEL = re.compile(r"[ؗ-ًؚ-ْٰـ]")  # diacritics + Tatweel

def normalize_ar(text, strip_diacritics=True, unify_alef=True,
                 ta_marbuta_to_ha=True, alef_maqsura_to_ya=True, unify_digits=True):
    if strip_diacritics:
        text = TASHKEEL.sub("", text)
    if unify_alef:
        text = re.sub("[آأإٱ]", "ا", text)  # آأإ ٱ -> ا
    if ta_marbuta_to_ha:
        text = text.replace("ة", "ه")  # ة -> ه
    if alef_maqsura_to_ya:
        text = text.replace("ى", "ي")  # ى -> ي
    if unify_digits:
        ar_digits = "٠١٢٣٤٥٦٧٨٩"
        text = text.translate({ord(a): str(i) for i, a in enumerate(ar_digits)})
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
def _levenshtein(ref, hyp):
    """Token-level edit distance with operation counts (S, D, I)."""
    n, m = len(ref), len(hyp)
    d = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): d[i][0] = i
    for j in range(m+1): d[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[n][m]

def error_rate(ref_tokens, hyp_tokens):
    if len(ref_tokens) == 0:
        return 0.0 if len(hyp_tokens) == 0 else 1.0
    return _levenshtein(ref_tokens, hyp_tokens) / len(ref_tokens)

def wer(ref, hyp):
    return error_rate(ref.split(), hyp.split())

def cer(ref, hyp):
    return error_rate(list(ref.replace(" ", "")), list(hyp.replace(" ", "")))

## 5. Score raw vs normalized, MSA vs dialect

In [ ]:
rows = []
for s in samples:
    ref, hyp = s['ref'], hyps[s['id']]
    raw_w = wer(ref, hyp)
    nref, nhyp = normalize_ar(ref), normalize_ar(hyp)
    norm_w = wer(nref, nhyp)
    norm_c = cer(nref, nhyp)
    rows.append((s['id'], s['dialect'], raw_w, norm_w, norm_c))

print(f"{'id':6} {'dialect':10} {'WER raw':>8} {'WER norm':>9} {'CER norm':>9}")
for r in rows:
    print(f'{r[0]:6} {r[1]:10} {r[2]*100:7.1f}% {r[3]*100:8.1f}% {r[4]*100:8.1f}%')

msa = [r for r in rows if r[1]=='MSA']
dia = [r for r in rows if r[1]!='MSA']
import statistics as st
print()
print('mean normalized WER  MSA    :', round(100*st.mean([r[3] for r in msa]),1), '%')
print('mean normalized WER  dialect:', round(100*st.mean([r[3] for r in dia]),1), '%')

## 6. Reflection (connects to Section 1.5)

Compare the MSA and dialect error rates above, then write two or three sentences:

1. How much does **normalization** alone change the MSA score? Why? (Hint: Alef and Ta-marbuta
   variants, diacritics.)
2. Why does a model trained mostly on MSA broadcast data **degrade on dialect**, even after
   normalization? Connect this to the data-imbalance and dialect-gap discussion in Section 1.5.
3. Which dialect suffered most here, and what kind of errors (substitutions toward MSA, deleted
   particles) drove the score?

## Exercise solutions

Chapter 1's exercises are mostly conceptual; one is solved here in code using the samples and
metrics already defined in this notebook.

**Exercise (MSA vs dialect error).** Using the offline samples and the normalization above, compute the normalized Word Error Rate per clip and the MSA-versus-dialect averages, and connect the gap to the data-imbalance discussion in Section 1.5.

In [ ]:
import statistics as st
rows=[]
for s in samples:
    nref,nhyp=normalize_ar(s['ref']),normalize_ar(hyps[s['id']])
    rows.append((s['dialect'], wer(nref,nhyp)))
msa=[w for d,w in rows if d=='MSA']; dia=[w for d,w in rows if d!='MSA']
print('normalized WER  MSA    :', round(100*st.mean(msa),1),'%')
print('normalized WER  dialect:', round(100*st.mean(dia),1),'%')
print('=> the dialect gap reflects MSA-heavy training data (Section 1.5).')

**Other exercises (conceptual).** Normalization mainly removes Alef and Ta-marbuta and diacritic differences, so it helps MSA most; dialect stays hard because the model was trained mostly on MSA, not because of spelling. Report which dialect suffered most and whether errors are substitutions toward MSA or dropped particles.